# DAGonStar scientific-workflow tutorial

This notebook is the executable companion to the canonical Lessons 00–17. It is designed for a local Jupyter kernel and Google Colab. Lessons 01–11 execute deterministic local examples. Lessons 12–14 execute structural tests because a hosted notebook cannot substantiate live Docker, SSH, or Slurm execution. Lessons 15–17 execute local FAIR, composition, and CWL checks.

For each lesson, state a prediction before running the cell, retain the output as evidence, and distinguish that evidence from claims about external infrastructure or scientific reproducibility.


## Lesson 00 — Establish the environment

The setup cell locates an existing checkout. When the notebook is opened as a standalone file in Colab, it clones the public repository into `/content/dagonstar`. It installs the checkout and Docker's optional Python client into the active kernel environment, then defines interpreter-safe helpers used by every later cell.


In [ ]:
import os
from pathlib import Path
import subprocess
import sys


def _find_repository():
    candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/dagonstar")]
    for candidate in candidates:
        if (candidate / "dagon").is_dir() and (candidate / "examples" / "tutorial").is_dir():
            return candidate.resolve()
    return None


ROOT = _find_repository()
try:
    import google.colab  # type: ignore[import-not-found]
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
if ROOT is None:
    if not IN_COLAB:
        raise RuntimeError("Open this notebook from a DAGonStar checkout, or set ROOT to that checkout.")
    ROOT = Path("/content/dagonstar")
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/DagOnStar/dagonstar.git", str(ROOT)],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[docker]"],
    cwd=ROOT,
    check=True,
)

RUN_ENV = os.environ.copy()
RUN_ENV["PYTHONPATH"] = str(ROOT) + os.pathsep + RUN_ENV.get("PYTHONPATH", "")


def run_example(relative_path):
    completed = subprocess.run(
        [sys.executable, str(ROOT / relative_path)],
        cwd=ROOT,
        env=RUN_ENV,
        check=True,
    )
    print(f"PASS: {relative_path}")
    return completed


def run_tests(*test_names):
    completed = subprocess.run(
        [sys.executable, "-m", "unittest", "-v", *test_names],
        cwd=ROOT,
        env=RUN_ENV,
        check=True,
    )
    print("PASS:", ", ".join(test_names))
    return completed


print("runtime:", "Google Colab" if IN_COLAB else "Jupyter/local")
print("python:", sys.version.split()[0])
print("repository:", ROOT)


## Lesson 01 — Create and run the first local task

Predict the output artifact and its value, then execute the credential-free local batch example.


In [ ]:
run_example("examples/tutorial/lesson_01_first_local_task.py")


## Lesson 02 — Build a directed acyclic graph

Predict the report task's two predecessors. The example verifies fan-out, fan-in, and completion order.


In [ ]:
run_example("examples/tutorial/lesson_02_build_a_dag.py")


## Lesson 03 — Connect tasks with workflow data references

Predict how the logical `workflow://` reference becomes a consumer-visible path and which dependency is discovered.


In [ ]:
run_example("examples/tutorial/lesson_03_workflow_data_dependencies.py")


## Lesson 04 — Validate and repair a cycle

Predict why the deliberate cycle has no topological order. The example verifies rejection and then validates the repaired graph without executing the invalid graph.


In [ ]:
run_example("examples/tutorial/lesson_04_validate_and_fix_cycles.py")


## Lesson 05 — Inspect scratch directories, launchers, and logs

Distinguish the scientific output from DAGonStar's operational records below `.dagon/` before running the example.


In [ ]:
run_example("examples/tutorial/lesson_05_scratch_launchers_and_logs.py")


## Lesson 06 — Compare data-staging modes

Hold the producer and consumer constant while predicting the filesystem difference between `COPY` and `LINK`.


In [ ]:
run_example("examples/tutorial/lesson_06_data_staging.py")


## Lesson 07 — Checkpoint and resume

Predict which completed task state can be reused and remember that checkpoint reuse does not validate undeclared external inputs.


In [ ]:
run_example("examples/tutorial/lesson_07_checkpoint_and_resume.py")


## Lesson 08 — Observe asynchronous lifecycle events

Predict the lifecycle event order. Joining the returned thread is required before treating the workflow as complete.


In [ ]:
run_example("examples/tutorial/lesson_08_asynchronous_execution.py")


## Lesson 09 — Execute a native Python function

Predict the structured JSON result produced by the importable Python target.


In [ ]:
run_example("examples/tutorial/lesson_09_native_python_tasks.py")


## Lesson 10 — Execute a web task against a local mock

The example binds a short-lived loopback server and makes no public request. Success demonstrates request construction and artifact handling, not production-service availability.


In [ ]:
run_example("examples/tutorial/lesson_10_web_tasks.py")


## Lesson 11 — Execute an LLM task against a local mock

The fixed local response isolates orchestration from provider cost, credentials, drift, and stochastic output. It does not evaluate a real model.


In [ ]:
run_example("examples/tutorial/lesson_11_llm_tasks.py")


## Lesson 12 — Verify the Docker task boundary

These mocked unit tests verify Docker task construction and command behavior without contacting a daemon. A passing cell is not evidence that Colab or the local host ran a container.


In [ ]:
run_tests("tests.test_docker_task")


## Lesson 13 — Verify the SSH task boundary

These tests verify safe configuration, quoting, launcher construction, and SSH policy without opening a network connection.


In [ ]:
run_tests("tests.test_security_configuration", "tests.test_shell_commands")


## Lesson 14 — Verify the Slurm task boundary

These focused tests inspect local Slurm command generation. They do not submit a job or establish cluster availability.


In [ ]:
run_tests(
    "tests.test_optional_integrations.OptionalIntegrationTests.test_slurm_command_includes_configured_scheduler_options",
    "tests.test_optional_integrations.OptionalIntegrationTests.test_slurm_command_quotes_all_user_controlled_values",
)


## Lesson 15 — Record FAIR metadata locally

The FAIR tests verify opt-in profiles, declarations, provenance, sanitisation, fixity, and local exports. They do not publish artifacts or mint identifiers.


In [ ]:
run_tests("tests.test_fair")


## Lesson 16 — Compose meta-workflows

The focused test verifies cross-workflow `workflow://` dependency discovery within a loaded local workflow set.


In [ ]:
run_tests("tests.test_workflow_core.WorkflowCoreTests.test_transversal_references_resolve_across_loaded_workflows")


## Lesson 17 — Export the command graph as CWL v1.2

The example test regenerates the checked-in CWL deterministically. If `cwltool` is installed, the standards-validation test also runs; otherwise it is reported as skipped. Backend-specific staging and remote execution remain outside the export boundary.


In [ ]:
run_tests("tests.test_cwl_example")


## Course verification

A complete run should leave every executable cell successful. The final validator checks the canonical lesson structure, cross-links, and notebook coverage. A pass supports only the local and structural claims stated above.


In [ ]:
run_example("tools/validate_tutorial.py")
